# A4 · DeepSeek-V4 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A4_deepseek_v4.ipynb)

원본: transformers `deepseek_v4` (순수 PyTorch). 내 구현: `miniarch/models/deepseek_v4.py`.

한 모델에 부품이 여럿 들어 있어서 통째로 검증합니다.
- attention: K = V 단일 head, sliding / CSA(압축 + indexer top-k) / HCA(강한 압축), head별 sink
- MoE: sqrt-softplus 점수 + top-k, 앞쪽 층은 토큰 id로 expert를 고정하는 hash 라우팅, shared expert
- mHC: residual을 여러 줄기로 들고 다니며 Sinkhorn으로 섞음

축소 config: 4층 = [sliding, CSA, HCA, CSA], window 8, CSA 2→1, HCA 8→1, indexer top-3, 시퀀스 40.
창 밖, 아직 다 안 찬 압축 칸, indexer가 고르지 않은 칸까지 모든 마스킹 경로가 쓰이도록 고른 값입니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 학습 동작 확인 → 6. 단계 전용 자체 점검 → 7. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A4")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('deepseek-ai/DeepSeek-V4-Flash', 'layers\\.0\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

### 원본과 비교할 수 없는 부분

- **수치 미분 (float64)**: 무작위 방향으로 가중치를 ±ε 움직여 잰 손실 변화가 autograd 기울기와 같은지 봅니다.
  원본이 추론 전용이라 기울기를 비교할 수 없는 모델도 이걸로 backward 를 확인합니다.
- **라우터 bias 업데이트 (aux-loss-free)**: 원본 추론 코드에는 없는 학습 전용 규칙입니다. 많이 뽑힌 expert 의 bias 는 내려가고
  적게 뽑힌 쪽은 올라가는지, bias 가 optimizer 에서 빠져 있는지, bias 가 실제로 선택을 바꾸는지 봅니다. 라우터 bias 가 없는 모델은 건너뜁니다.

In [ ]:
try:
    f = TR.directional_grad_check(pair["mine"], pair, inputs, dev)
    print(f"방향 {len(f.rel_errors)}개 상대 오차 {[f'{e:.1e}' for e in f.rel_errors]}  (ε={f.eps})")
    report("수치 미분 == autograd (float64)", f.ok, max(f.rel_errors))
except Exception:
    traceback.print_exc(limit=4); report("수치 미분 == autograd (float64)", False)
try:
    b = TR.router_bias_check(pair["mine"], pair, inputs, dev)
    print("\n".join(b.details))
    report("라우터 bias 업데이트" if b.n_routers else "라우터 bias 업데이트 (라우터 bias 없음 → 건너뜀)", b.ok)
except Exception:
    traceback.print_exc(limit=4); report("라우터 bias 업데이트", False)

## 단계 전용 자체 점검

### attention map · sink · 라우팅 hook 확인

- `attn_probs`: [B, H, T, T + 압축 칸]. 각 행 합 + sink 확률 = 1이어야 합니다.
- `sink_probe`: sink로 간 확률. `route_probe`: 토큰마다 고른 expert.

In [ ]:
from miniarch.models import deepseek_v4 as D
ref, _ = S.ref()
m = D.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "sink_probe"))]
x = torch.randint(0, V.cases.VOCAB, (2, 40), device=dev)
with V.capture(m, names) as st, torch.no_grad():
    m(x)
for i in range(m.config.n_layer):
    p, s = st[f"layers.{i}.self_attn.attn_probs"], st[f"layers.{i}.self_attn.sink_probe"]
    err = (p.sum(-1) + s - 1).abs().max().item()
    print(f"layer {i} {m.config.layer_types[i]:<32} probs={tuple(p.shape)}  행합+sink 오차={err:.1e}  평균 sink={s.mean():.3f}")
    report(f"layer {i} 확률 합", err < 1e-5, err)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A4_deepseek_v4.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A4_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")